# Proyecto: Predicción de Incumplimiento de SLA en Sistema de Soporte Técnico
## Fase 1: Recopilación y Transformación de Datos

**Fuente de datos:** Base de datos Supabase - Sistema Lillosupport  
**Tablas utilizadas:** `incidencias`, `ans_resultados`, `ans_configuracion`, `tipos_servicio`  
**Objetivo:** Preparar un dataset limpio y enriquecido para predecir si una incidencia cumplirá o no el ANS.

## Carga del Dataset



In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataset
df = pd.read_csv('BD.csv')

print(f'Columnas: {df.shape[1]}')
df.head()

In [ ]:
# Resumen general del dataset
print('--Tipos de datos')
print(df.dtypes)
print()
print('--Valores nulos por columna')
print(df.isnull().sum())

## Limpieza de las columnas

In [ ]:
# Columnas de tipo fecha
columnas_fecha = [
    'created_at', 'updated_at', 'fecha_creacion',
    'fecha_limite_ans', 'ans_fecha_limite', 'ans_fecha_cierre'
]

for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

print('Fechas convertidas correctamente.')
df[columnas_fecha].dtypes

### Creación de la variable objetivo: `cumple_ans`

Esta es la variable que el modelo predictivo intentará predecir.  
- `1` Significaque la incidencia fue resuelta antes del límite de tiempo (cumplió el ANS)  
- `0` Significa que la incidencia superó el tiempo límite (incumplió el ANS)

In [ ]:
# Variable objetivo basada en ans_estado_final
df['cumple_ans'] = df['ans_estado_final'].apply(
    lambda x: 1 if str(x).strip().lower() == 'cumplido' else 0
)

print('Distribución de la variable objetivo:')
print(df['cumple_ans'].value_counts())
print()
print(f"Tasa de cumplimiento: {df['cumple_ans'].mean()*100:.1f}%")

### Creación de variables derivadas

In [ ]:
# Tiempo real de resolución en horas (desde creación hasta cierre)
df['horas_resolucion'] = (
    df['ans_fecha_cierre'] - df['fecha_creacion']
).dt.total_seconds() / 3600

# Tiempo límite en horas (desde creación hasta límite ANS)
df['horas_limite_ans'] = (
    df['ans_fecha_limite'] - df['fecha_creacion']
).dt.total_seconds() / 3600

# Horas de diferencia entre cierre y límite que signifia negativo = cumplió, positivo = venció
df['horas_diferencia_ans'] = (
    df['ans_fecha_cierre'] - df['ans_fecha_limite']
).dt.total_seconds() / 3600

# Porcentaje de tiempo consumido respecto al total
df['pct_tiempo_consumido'] = (
    df['minutos_consumidos'] / df['minutos_totales'].replace(0, np.nan)
) * 100

# Variables de fecha: mes, día de semana, hora de creación
df['mes_creacion'] = df['fecha_creacion'].dt.month
df['dia_semana_creacion'] = df['fecha_creacion'].dt.dayofweek  # 0=Lunes, 6=Domingo
df['hora_creacion'] = df['fecha_creacion'].dt.hour

# Turno del día
def turno(hora):
    if 6 <= hora < 12:
        return 'mañana'
    elif 12 <= hora < 18:
        return 'tarde'
    elif 18 <= hora < 24:
        return 'noche'
    else:
        return 'madrugada'

df['turno_creacion'] = df['hora_creacion'].apply(turno)

print('Variables derivadas creadas correctamente.')
df[['horas_resolucion','horas_limite_ans','horas_diferencia_ans',
    'pct_tiempo_consumido','mes_creacion','dia_semana_creacion',
    'hora_creacion','turno_creacion']].head()

### Limpieza de columnas

In [ ]:
# Limpiar columna ans_tiempo_limite pasar de '5d' a 5
df['dias_limite_config'] = df['ans_tiempo_limite'].str.extract(r'(\d+)').astype(float)

# Estandarizar texto en columnas categóricas
for col in ['estado', 'prioridad', 'tipo_servicio', 'ans_estado', 'ans_estado_final', 'turno_creacion']:
    df[col] = df[col].str.strip().str.lower()

# Columna finalizado a booleano numérico
df['finalizado'] = df['finalizado'].map({'true': 1, 'false': 0, True: 1, False: 0})

print('Limpieza completada.')
print(f'Valores nulos restantes:')
print(df.isnull().sum()[df.isnull().sum() > 0])

### Tratamiento de valores nulos

In [ ]:
# Rellenar nulos en columnas numéricas con la mediana
cols_numericas = ['horas_resolucion', 'horas_limite_ans', 'horas_diferencia_ans',
                  'pct_tiempo_consumido', 'costo_base', 'descuento_ans']

# Guardar máscara antes de rellenar
mask_nulos = df['ans_fecha_cierre'].isnull()

for col in cols_numericas:
    if df[col].isnull().sum() > 0:
        mediana = df[col].median()
        df[col].fillna(mediana, inplace=True)
        print(f'{col}: rellenado con mediana ({mediana:.2f})')

print('\n--Nulos despues del proceo')
for col in cols_numericas:
    print(f'{col}: {df[col].isnull().sum()} nulos restantes')

print(f'\n--Filas que fueron rellenadas: {mask_nulos.sum()}')
df[mask_nulos][['incidencia_id', 'horas_resolucion', 'horas_limite_ans',
                'horas_diferencia_ans', 'pct_tiempo_consumido']]

## Dataset final limpio

In [ ]:
# Resultado
columnas_finales = [
    'incidencia_id',
    'fecha_creacion',
    'estado',
    'prioridad',
    'tipo_servicio',
    'ans_estado',
    'ans_estado_final',
    'cumple_ans',
    'horas_resolucion',
    'horas_limite_ans',
    'horas_diferencia_ans',
    'pct_tiempo_consumido',
    'minutos_consumidos',
    'minutos_totales',
    'minutos_restantes',
    'dias_limite_config',
    'ans_penalidad_porcentaje',
    'costo_base_servicio',
    'penalidad_porcentaje',
    'costo_servicio',
    'mes_creacion',
    'dia_semana_creacion',
    'hora_creacion',
    'turno_creacion',
    'finalizado',
    'billing_status'
]

df_final = df[columnas_finales].copy()

# Guardar dataset limpio
df_final.to_csv('BD_limpia.csv', index=False)

print('=== DATASET FINAL ===')
print(f'Registros: {df_final.shape[0]}')
print(f'Columnas: {df_final.shape[1]}')
print(f'Cumple ANS: {df_final["cumple_ans"].sum()} ({df_final["cumple_ans"].mean()*100:.1f}%)')
print(f'Incumple ANS: {(df_final["cumple_ans"]==0).sum()} ({(1-df_final["cumple_ans"].mean())*100:.1f}%)')
print()
df_final.head(10)

In [ ]:
# Estadísticas descriptivas del dataset final
df_final.describe()

## Fase 2: Análisis exploratorio de datos

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

df = pd.read_csv('BD_limpia.csv', parse_dates=['fecha_creacion'])

print(f'Registros: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head()

## Variable Objetivo: cumple_ans


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
conteo = df['cumple_ans'].value_counts()
etiquetas = ['Incumplió ANS\n(0)', 'Cumplió ANS\n(1)']
colores = ['#e74c3c', '#2ecc71']
axes[0].bar(etiquetas, [conteo[0], conteo[1]], color=colores, edgecolor='black', width=0.5)
axes[0].set_title('Distribución de Cumplimiento de ANS', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad de Incidencias')
for i, v in enumerate([conteo[0], conteo[1]]):
    axes[0].text(i, v + 0.5, f'{v} ({v}%)', ha='center', fontweight='bold')

# Gráfico de torta
axes[1].pie([conteo[0], conteo[1]], labels=['Incumplió\nANS', 'Cumplió\nANS'],
            colors=colores, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Proporción de Cumplimiento', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_cumple_ans.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dataset perfectamente balanceado: 50% cumple / 50% incumple')

Analisis por prioridad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribución de prioridad
prioridad_conteo = df['prioridad'].value_counts()
axes[0].bar(prioridad_conteo.index, prioridad_conteo.values,
            color=['#3498db','#e67e22','#e74c3c'], edgecolor='black')
axes[0].set_title('Distribución por Prioridad', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(prioridad_conteo.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Cumplimiento ANS por prioridad — conteos absolutos
pivot = df.groupby('prioridad')['cumple_ans'].value_counts().unstack(fill_value=0)
pivot.columns = ['Incumplió', 'Cumplió']

x = range(len(pivot))
width = 0.35
axes[1].bar([i - width/2 for i in x], pivot['Cumplió'], width=width,
            color='#2ecc71', edgecolor='black', label='Cumplió')
axes[1].bar([i + width/2 for i in x], pivot['Incumplió'], width=width,
            color='#e74c3c', edgecolor='black', label='Incumplió')
axes[1].set_title('Cumplimiento ANS por Prioridad (conteo)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cantidad de Incidencias')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(pivot.index)
axes[1].legend()
for i, (c, nc) in enumerate(zip(pivot['Cumplió'], pivot['Incumplió'])):
    axes[1].text(i - width/2, c + 0.1, str(c), ha='center', fontweight='bold')
    axes[1].text(i + width/2, nc + 0.1, str(nc), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_prioridad.png', dpi=150, bbox_inches='tight')
plt.show()

# Hipotesis del análisis

"Variable objetivo: cumple_ans (1 = cumplió el ANS, 0 = incumplió)

HIPÓTESIS PRINCIPAL:
Es posible predecir si una incidencia cumplirá o no el ANS
antes de que venza el tiempo límite, usando características
conocidas en el momento de su creación.

HIPÓTESIS ESPECÍFICAS:

H1 — Prioridad:
Las incidencias de prioridad BAJA tienen mayor probabilidad
de incumplir el ANS porque al no ser urgentes se atienden
con menor inmediatez.

H2 — Tipo de servicio:
El tipo de servicio influye en el cumplimiento del ANS ya que
algunos servicios requieren más tiempo logístico que otros
(ej. cambio de activo vs ingreso de empleado).

H3 — Turno de creación:
Las incidencias creadas en horario de NOCHE o MADRUGADA
tienen mayor probabilidad de incumplir el ANS porque se
atienden al día siguiente, consumiendo tiempo del límite.

H4 — Tiempo límite configurado:
A menor tiempo límite asignado (dias_limite_config),
mayor riesgo de incumplimiento porque hay menos margen
para la operación logística.

H5 — Día de la semana:
Las incidencias creadas los VIERNES tienen mayor riesgo
de incumplir porque el fin de semana reduce la operación.
"

print("Hipótesis definidas correctamente.")
print("Variable objetivo: cumple_ans")
print(f"Distribución: {df['cumple_ans'].value_counts().to_dict()}")

# Análisis por Tipo de Servicio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Orden consistente por cantidad
orden = df['tipo_servicio'].value_counts().index.tolist()

# Gráfica izquierda — Cantidad por tipo de servicio
tipo_conteo = df['tipo_servicio'].value_counts().reindex(orden)
axes[0].barh(tipo_conteo.index, tipo_conteo.values,
             color=sns.color_palette('Set2', len(tipo_conteo)), edgecolor='black')
axes[0].set_title('Incidencias por Tipo de Servicio', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cantidad')
for i, v in enumerate(tipo_conteo.values):
    axes[0].text(v + 0.2, i, str(v), va='center', fontweight='bold')

# Gráfica derecha — Cumplimiento en el mimo orden
pivot_tipo = df.groupby('tipo_servicio')['cumple_ans'].mean() * 100
pivot_tipo = pivot_tipo.reindex(orden)
colores_tipo = ['#e74c3c' if v < 50 else '#2ecc71' for v in pivot_tipo.values]
axes[1].barh(pivot_tipo.index, pivot_tipo.values, color=colores_tipo, edgecolor='black')
axes[1].set_title('Tasa de Cumplimiento ANS por Tipo de Servicio (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('% Cumplimiento')
axes[1].axvline(50, color='gray', linestyle='--', alpha=0.7)
for i, v in enumerate(pivot_tipo.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_tipo_servicio.png', dpi=150, bbox_inches='tight')
plt.show()

## Análisis Temporal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']

# Incidencias por día de la semana
dia_conteo = df['dia_semana_creacion'].dropna().astype(int).value_counts().sort_index()
axes[0].bar([dias[i] for i in dia_conteo.index], dia_conteo.values,
            color=sns.color_palette('Blues_d', len(dia_conteo)), edgecolor='black')
axes[0].set_title('Incidencias por Día de la Semana', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(dia_conteo.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Cumplimiento ANS por turno del día
pivot_turno = df.groupby('turno_creacion')['cumple_ans'].mean() * 100
colores_turno = ['#e74c3c' if v < 50 else '#2ecc71' for v in pivot_turno.values]
axes[1].bar(pivot_turno.index, pivot_turno.values, color=colores_turno, edgecolor='black')
axes[1].set_title('Tasa de Cumplimiento ANS por Turno', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% Cumplimiento')
axes[1].set_ylim(0, 110)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.7)
for i, v in enumerate(pivot_turno.values):
    axes[1].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

## Distribución de Tiempos de Resolución

In [ ]:
# Distribución de días de la semana en que se cierran las incidencias
# Usamos fecha_creacion + horas_resolucion para reconstruir la fecha de cierre
df['fecha_cierre_calc'] = pd.to_datetime(df['fecha_creacion'], utc=True, errors='coerce') + \
                          pd.to_timedelta(df['horas_resolucion'], unit='h')

df['dia_semana_cierre'] = df['fecha_cierre_calc'].dt.dayofweek

dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Todos los cierres por día
cierre_conteo = df['dia_semana_cierre'].dropna().astype(int).value_counts().sort_index()
axes[0].bar([dias[i] for i in cierre_conteo.index], cierre_conteo.values,
            color=sns.color_palette('Blues_d', len(cierre_conteo)), edgecolor='black')
axes[0].set_title('Incidencias Cerradas por Día de la Semana', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(cierre_conteo.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Cierres por día separados por cumplimiento
for val, color, label in [(1, '#2ecc71', 'Cumplió'), (0, '#e74c3c', 'Incumplió')]:
    sub = df[df['cumple_ans'] == val]['dia_semana_cierre'].dropna().astype(int).value_counts().sort_index()
    axes[1].plot([dias[i] for i in sub.index], sub.values,
                 marker='o', color=color, label=label, linewidth=2)

axes[1].set_title('Cierres por Día según Cumplimiento ANS', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cantidad')
axes[1].legend()

plt.tight_layout()
plt.savefig('grafico_cierre_dias.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución de días de la semana en que se cierran las incidencias
df['fecha_cierre_calc'] = pd.to_datetime(df['fecha_creacion'], utc=True, errors='coerce') + \
                          pd.to_timedelta(df['horas_resolucion'], unit='h')
df['dia_semana_cierre'] = df['fecha_cierre_calc'].dt.dayofweek

dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfica 1 — Incidencias creadas por día
dia_conteo = df['dia_semana_creacion'].dropna().astype(int).value_counts().sort_index()
axes[0].bar([dias[i] for i in dia_conteo.index], dia_conteo.values,
            color=sns.color_palette('Blues_d', len(dia_conteo)), edgecolor='black')
axes[0].set_title('Incidencias Creadas\npor Día de la Semana', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(dia_conteo.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Gráfica 2 — Incidencias cerradas por día
cierre_conteo = df['dia_semana_cierre'].dropna().astype(int).value_counts().sort_index()
axes[1].bar([dias[i] for i in cierre_conteo.index], cierre_conteo.values,
            color=sns.color_palette('Oranges_d', len(cierre_conteo)), edgecolor='black')
axes[1].set_title('Incidencias Cerradas\npor Día de la Semana', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cantidad')
for i, v in enumerate(cierre_conteo.values):
    axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Gráfica 3 — Cierres por día según cumplimiento
for val, color, label in [(1, '#2ecc71', 'Cumplió'), (0, '#e74c3c', 'Incumplió')]:
    sub = df[df['cumple_ans'] == val]['dia_semana_cierre'].dropna().astype(int).value_counts().sort_index()
    axes[2].plot([dias[i] for i in sub.index], sub.values,
                 marker='o', color=color, label=label, linewidth=2)
axes[2].set_title('Cierres por Día según\nCumplimiento ANS', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Cantidad')
axes[2].legend()

plt.suptitle('Análisis Temporal de Incidencias', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_temporal_completo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
resumen = {
    'Balance del dataset': '50% cumple / 50% incumple - ideal para modelado',
    'Prioridad dominante': '91% de incidencias son de prioridad baja',
    'Tipo de servicio mas frecuente': 'Cambio de activo (41 casos)',
    'Todas entregadas': '100% de las incidencias tiene estado entregado',
    'Variables mas correlacionadas': 'horas_diferencia_ans y pct_tiempo_consumido',
    'Patron temporal': 'El turno y dia de la semana influyen en el cumplimiento',
    'Servicios con mas incumplimiento': 'Activo en venta (25%) y Cambio de activo (34.1%)',
    'Prioridad mas problematica': 'Baja prioridad con 48.4% de cumplimiento'
}

print('=' * 60)
print('RESUMEN DE HALLAZGOS DEL EDA')
print('=' * 60)
for k, v in resumen.items():
    print(f'- {k}: {v}')

print()

## Fase 3: Inteligencia de negocios

In [ ]:
# Tabla principal de hechos
df_hechos = df[[
    'incidencia_id',
    'fecha_creacion',
    'estado',
    'prioridad',
    'tipo_servicio',
    'cumple_ans',
    'horas_resolucion',
    'horas_limite_ans',
    'horas_diferencia_ans',
    'pct_tiempo_consumido',
    'minutos_consumidos',
    'minutos_totales',
    'costo_base_servicio',
    'penalidad_porcentaje',
    'costo_servicio',
    'billing_status',
    'mes_creacion',
    'dia_semana_creacion',
    'hora_creacion',
    'turno_creacion',
    'dias_limite_config'
]].copy()

# Tabla dimensión prioridad
df_dim_prioridad = pd.DataFrame({
    'prioridad': ['baja', 'alta', 'critica'],
    'nivel_urgencia': [1, 2, 3],
    'descripcion': ['Sin urgencia inmediata', 'Requiere atención pronta', 'Atención inmediata requerida']
})

# Tabla dimensión tipo de servicio
df_dim_servicio = df[['tipo_servicio', 'dias_limite_config',
                       'ans_penalidad_porcentaje']].drop_duplicates().reset_index(drop=True)

# Tabla dimensión tiempo
df_dim_tiempo = df[['fecha_creacion', 'mes_creacion',
                    'dia_semana_creacion', 'hora_creacion',
                    'turno_creacion']].drop_duplicates().reset_index(drop=True)

dias_nombre = {0:'Lunes', 1:'Martes', 2:'Miercoles',
               3:'Jueves', 4:'Viernes', 5:'Sabado', 6:'Domingo'}
meses_nombre = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril',
                5:'Mayo', 6:'Junio', 7:'Julio', 8:'Agosto',
                9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}

df_dim_tiempo['nombre_dia'] = df_dim_tiempo['dia_semana_creacion'].map(dias_nombre)
df_dim_tiempo['nombre_mes'] = df_dim_tiempo['mes_creacion'].map(meses_nombre)

# Exportar todas las tablas como CSV para Power BI
df_hechos.to_csv('BI_hechos_incidencias.csv', index=False)
df_dim_prioridad.to_csv('BI_dim_prioridad.csv', index=False)
df_dim_servicio.to_csv('BI_dim_servicio.csv', index=False)
df_dim_tiempo.to_csv('BI_dim_tiempo.csv', index=False)

print('Tablas exportadas para Power BI:')
print(f'- BI_hechos_incidencias.csv: {df_hechos.shape[0]} filas x {df_hechos.shape[1]} cols')
print(f'- BI_dim_prioridad.csv: {df_dim_prioridad.shape[0]} filas')
print(f'- BI_dim_servicio.csv: {df_dim_servicio.shape[0]} filas')
print(f'- BI_dim_tiempo.csv: {df_dim_tiempo.shape[0]} filas')